In [1]:
!pip install pythainlp -q

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import ClassifierChain
from sklearn.metrics import f1_score
from pythainlp.tokenize import word_tokenize
import joblib

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 68.9 MB/s eta 0:00:00
/kaggle/input/datasets/yoksamutr/midterm/sample_submission.csv
/kaggle/input/datasets/yoksamutr/midterm/train.csv
/kaggle/input/datasets/yoksamutr/midterm/test.csv


In [2]:
# 1. Load Data
train_df = pd.read_csv('/kaggle/input/datasets/yoksamutr/midterm/train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/yoksamutr/midterm/test.csv')

label_cols = [
    'สำนักงานตำรวจแห่งชาติ', 'การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย', 
    'สภาเด็กและเยาวชนกรุงเทพมหานคร', 'กรมควบคุมมลพิษ', 'กรมสรรพสามิต', 
    'การไฟฟ้านครหลวง', 'กรมทางหลวง', 'สำนักงานประกันสุขภาพแห่งชาติ', 
    'การประปานครหลวง', 'คณะกรรมการการพัฒนาเศรษฐกิจ', 
    'กระทรวงการท่องเที่ยวและกีฬา', 'สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200'
]

train_df['comment'] = train_df['comment'].fillna("ไม่มีรายละเอียด")
test_df['comment'] = test_df['comment'].fillna("ไม่มีรายละเอียด")

In [3]:
# 2. Vectorization (Fit on ALL training data)
print("Vectorizing Text...")
def thai_tokenizer(text):
    return word_tokenize(text, engine='newmm', keep_whitespace=False)

vectorizer = TfidfVectorizer(
    tokenizer=thai_tokenizer, 
    token_pattern=None,
    ngram_range=(1, 2), 
    max_features=20000, 
    max_df=0.85
)

X_train_tfidf = vectorizer.fit_transform(train_df['comment'])
y_train = train_df[label_cols].values
X_test_tfidf = vectorizer.transform(test_df['comment'])

# Save the fitted vectorizer
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')
print("Saved TF-IDF Vectorizer to 'tfidf_vectorizer.joblib'")

Vectorizing Text...
Saved TF-IDF Vectorizer to 'tfidf_vectorizer.joblib'


In [4]:
# 3. Setup 5-Fold Cross Validation
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Matrices to store our Out-Of-Fold (OOF) predictions and Test predictions
oof_probs = np.zeros_like(y_train, dtype=float)
test_probs_sum = np.zeros((X_test_tfidf.shape[0], len(label_cols)), dtype=float)

print(f"Starting {N_SPLITS}-Fold Cross Validation...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tfidf)):
    print(f"--- Training Fold {fold + 1} ---")
    
    X_tr, y_tr = X_train_tfidf[train_idx], y_train[train_idx]
    X_va, y_va = X_train_tfidf[val_idx], y_train[val_idx]
    
    # Base model + Classifier Chain
    base_lr = LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=200)
    model = ClassifierChain(base_lr, order='random', random_state=42)
    
    # Train
    model.fit(X_tr, y_tr)

    # NEW: Save this specific fold's trained model
    joblib.dump(model, f'classifier_chain_fold_{fold + 1}.joblib')
    
    # Predict on Validation Fold (Save to our OOF matrix)
    oof_probs[val_idx] = model.predict_proba(X_va)
    
    # Predict on Test Data (Accumulate probabilities for ensembling later)
    test_probs_sum += model.predict_proba(X_test_tfidf)

# Average the test predictions across all 5 models
final_test_probs = test_probs_sum / N_SPLITS

Starting 5-Fold Cross Validation...
--- Training Fold 1 ---
--- Training Fold 2 ---
--- Training Fold 3 ---
--- Training Fold 4 ---
--- Training Fold 5 ---


In [5]:
# 4. Tune Thresholds on the Complete OOF Matrix
print("\nTuning optimal thresholds across all Out-Of-Fold predictions...")
best_thresholds = []
for i in range(len(label_cols)):
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.1, 0.9, 0.05):
        preds = (oof_probs[:, i] > thresh).astype(int)
        score = f1_score(y_train[:, i], preds, average='binary', zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh
    best_thresholds.append(best_thresh)

# Calculate final True OOF Macro F1
oof_final_preds = np.zeros_like(oof_probs)
for i in range(len(label_cols)):
    oof_final_preds[:, i] = (oof_probs[:, i] > best_thresholds[i]).astype(int)

true_macro_f1 = f1_score(y_train, oof_final_preds, average='macro', zero_division=0)
print(f"\n✅ YOUR TRUE OOF MACRO F1 SCORE IS: {true_macro_f1:.4f}")

# Save the best thresholds list
joblib.dump(best_thresholds, 'best_thresholds.joblib')
print("Saved optimal thresholds to 'best_thresholds.joblib'")


Tuning optimal thresholds across all Out-Of-Fold predictions...

✅ YOUR TRUE OOF MACRO F1 SCORE IS: 0.3325
Saved optimal thresholds to 'best_thresholds.joblib'


In [6]:
# 5. Apply thresholds to averaged test predictions and save
final_test_preds = np.zeros_like(final_test_probs)
for i in range(len(label_cols)):
    final_test_preds[:, i] = (final_test_probs[:, i] > best_thresholds[i]).astype(int)

sub_df = pd.DataFrame(final_test_preds, columns=label_cols)
sub_df.insert(0, 'id', test_df['id'])
sub_df.to_csv('kfold_chain_submission.csv', index=False)
print("Saved robust submission to 'kfold_chain_submission.csv'")

Saved robust submission to 'kfold_chain_submission.csv'


In [7]:
train_df.to_csv('prepared_train.csv', index=False, encoding='utf-8-sig')
test_df.to_csv('prepared_test.csv', index=False, encoding='utf-8-sig')
print("Prepared data saved as 'prepared_train.csv' and 'prepared_test.csv'")

Prepared data saved as 'prepared_train.csv' and 'prepared_test.csv'
